# Hugging Face Chapter 3 Lab: Fine-Tuning a Pretrained Model

This notebook combines the main examples from Hugging Face LLM Course Chapter 3.1–3.6.

Main topic:

> Fine-tune a pretrained BERT model on the GLUE MRPC dataset.

We will cover:

1. What GLUE and MRPC are
2. Loading a dataset from Hugging Face Datasets
3. Tokenizing sentence pairs
4. Dynamic padding with `DataCollatorWithPadding`
5. Fine-tuning with `Trainer`
6. Evaluating with accuracy and F1
7. Saving the model
8. Using the fine-tuned model for inference
9. Optional: what a manual PyTorch training loop looks like

Recommended environment: Google Colab with GPU, or a local Python environment with compatible `torch`, `transformers`, `datasets`, and `evaluate`.


## 0. Install packages

Run this cell in Google Colab or a fresh virtual environment.

If you already installed these packages, you may skip this cell.


In [ ]:
# For Google Colab or a clean environment
# %pip install -q transformers datasets evaluate accelerate sentencepiece
print("All required libraries are installed."))

## 1. What are GLUE and MRPC?

### GLUE

**GLUE** stands for **General Language Understanding Evaluation**.

It is a benchmark collection of NLP tasks used to evaluate how well language models understand text. It includes multiple tasks such as sentiment classification, paraphrase detection, textual entailment, and linguistic acceptability.

In Hugging Face Datasets, we can load tasks from GLUE like this:

```python
load_dataset("glue", "mrpc")
```

### MRPC

**MRPC** stands for **Microsoft Research Paraphrase Corpus**.

It is a sentence-pair classification dataset. Each example contains two sentences and a label:

| Label | Meaning |
|---|---|
| `0` | the two sentences are **not equivalent** |
| `1` | the two sentences are **equivalent / paraphrases** |

So the task is:

> Given two sentences, predict whether they mean the same thing.


## 2. Import libraries

In [ ]:
import numpy as np
import torch

from datasets import load_dataset
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    pipeline,
)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## 3. Load the MRPC dataset from GLUE

The dataset contains three splits: `train`, `validation`, and `test`. For this lab, we train on `train` and evaluate on `validation`.


In [ ]:
raw_datasets = load_dataset("glue", "mrpc")
raw_datasets

## 4. Inspect one example

Each example has `sentence1`, `sentence2`, `label`, and `idx`.


In [ ]:
raw_train_dataset = raw_datasets["train"]
example = raw_train_dataset[0]
example

In [ ]:
print("Sentence 1:", example["sentence1"])
print("Sentence 2:", example["sentence2"])
print("Label:", example["label"])

print("
Dataset features:")
raw_train_dataset.features

## 5. Load tokenizer

We use `bert-base-uncased`, a pretrained BERT model.

Important:

> The tokenizer and the model should usually come from the same checkpoint.


In [ ]:
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


## 6. Tokenize one sentence pair

For MRPC, the input is a pair of sentences.

BERT expects the structure:

```text
[CLS] sentence1 [SEP] sentence2 [SEP]
```

The tokenizer also creates `token_type_ids`, which tell BERT which tokens belong to the first sentence and which belong to the second sentence.


In [ ]:
sentence1 = "This is the first sentence."
sentence2 = "This is the second one."

inputs = tokenizer(sentence1, sentence2)
inputs

In [ ]:
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"])

for token, token_type_id in zip(tokens, inputs["token_type_ids"]):
    print(f"{token:12s} token_type_id={token_type_id}")


### Explanation

For BERT:

- `token_type_id = 0` means sentence 1
- `token_type_id = 1` means sentence 2

Some models, such as DistilBERT, do not use `token_type_ids`. That is fine. The tokenizer knows what the selected model needs.


## 7. Tokenize the full dataset

We define a function and apply it to the dataset using `.map()`.

`batched=True` means the tokenizer receives many examples at once, which is faster than processing one example at a time.


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["sentence1"],
        examples["sentence2"],
        truncation=True,
    )

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

## 8. Dynamic padding with `DataCollatorWithPadding`

There are two common padding strategies:

### Static padding

Pad every example to a fixed maximum length.

```python
padding="max_length"
```

This is simple but may waste memory.

### Dynamic padding

Pad examples only to the longest sequence in the current batch.

```python
DataCollatorWithPadding(tokenizer=tokenizer)
```

This is more efficient for many NLP tasks.


In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}

batch = data_collator(samples)

{k: v.shape for k, v in batch.items()}

## 9. Load a model for sequence classification

We use `AutoModelForSequenceClassification`.

This loads:

```text
BERT base model + classification head
```

For MRPC, we need two output labels:

```text
0 = not equivalent
1 = equivalent
```

The warning about newly initialized weights is normal: the classification head is new and will be learned during fine-tuning.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)


## 10. Define evaluation metrics

MRPC is commonly evaluated with accuracy and F1 score.

The model outputs logits. We convert logits into predicted class IDs by taking `argmax`.


In [ ]:
metric = evaluate.load("glue", "mrpc")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)


## 11. Fine-tune with `Trainer`

`TrainingArguments` defines the training settings.

`Trainer` runs the training loop for us.

For classroom demo, we use small settings:

- 1 epoch
- small batch size
- evaluation at the end of each epoch
- no external logging tools


In [ ]:
training_args = TrainingArguments(
    output_dir="./bert_mrpc_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


## 12. Start fine-tuning

This step updates the model weights.

Important wording:

> We are not training BERT from scratch. We are fine-tuning a pretrained BERT model on MRPC.


In [ ]:
trainer.train()

## 13. Evaluate the fine-tuned model

After training, we evaluate on the validation set.


In [ ]:
eval_results = trainer.evaluate()
eval_results

## 14. Inspect logits and predictions

`Trainer.predict()` returns raw logits, label IDs, and metrics.

For MRPC, each example has two logits:

```text
[score_for_not_equivalent, score_for_equivalent]
```


In [ ]:
predictions = trainer.predict(tokenized_datasets["validation"])

print("Logits shape:", predictions.predictions.shape)
print("Labels shape:", predictions.label_ids.shape)

predicted_labels = np.argmax(predictions.predictions, axis=-1)

print("First 10 predicted labels:", predicted_labels[:10])
print("First 10 true labels:     ", predictions.label_ids[:10])


## 15. Save the fine-tuned model

This saves both model weights/configuration and tokenizer files. After saving, the model can be loaded again with `pipeline`.


In [ ]:
save_dir = "./bert_mrpc_finetuned"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("Model saved to:", save_dir)


## 16. Use the saved model with `pipeline`

For a sentence-pair classification model, use:

```python
pipeline("text-classification", model=save_dir, tokenizer=save_dir)
```

The pipeline returns a label and confidence score.

Depending on the model configuration, labels may appear as `LABEL_0` and `LABEL_1`.

For this dataset:

```text
LABEL_0 = not equivalent
LABEL_1 = equivalent
```


In [ ]:
classifier = pipeline(
    "text-classification",
    model=save_dir,
    tokenizer=save_dir,
)

test_pairs = [
    {
        "text": "The company released a new product today.",
        "text_pair": "A new product was launched by the company today.",
    },
    {
        "text": "The weather is sunny today.",
        "text_pair": "The stock market closed lower yesterday.",
    },
]

for pair in test_pairs:
    result = classifier(pair["text"], text_pair=pair["text_pair"])
    print("Sentence 1:", pair["text"])
    print("Sentence 2:", pair["text_pair"])
    print("Prediction:", result)
    print("-" * 80)


## 17. Learning curves: what should students look for?

During training, useful signs include:

### Healthy learning

- training loss decreases
- validation accuracy/F1 improves
- validation loss does not increase strongly

### Possible overfitting

- training loss keeps decreasing
- validation loss increases
- training accuracy is much higher than validation accuracy

### Possible learning-rate issue

- loss is unstable or jumps a lot: learning rate may be too high
- loss barely changes: learning rate may be too low

In this short classroom run, the goal is not to get the best score. The goal is to understand the fine-tuning workflow.


In [ ]:
# View trainer logs after training
trainer.state.log_history[:5]

## 18. Optional: Prepare data manually for a PyTorch training loop

The `Trainer` API hides many details. The manual training loop helps students understand what happens internally.

The manual loop needs extra preprocessing:

1. remove text columns the model does not use
2. rename `label` to `labels`
3. set format to PyTorch tensors
4. create DataLoaders


In [ ]:
from torch.utils.data import DataLoader

manual_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
manual_datasets = manual_datasets.rename_column("label", "labels")
manual_datasets.set_format("torch")

train_dataloader = DataLoader(
    manual_datasets["train"],
    shuffle=True,
    batch_size=8,
    collate_fn=data_collator,
)

eval_dataloader = DataLoader(
    manual_datasets["validation"],
    batch_size=8,
    collate_fn=data_collator,
)

for batch in train_dataloader:
    break

{k: v.shape for k, v in batch.items()}


## 19. Optional: One forward pass manually

When `labels` are included, the model returns both loss and logits. This is what makes training possible.


In [ ]:
manual_model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

outputs = manual_model(**batch)

print("Loss:", outputs.loss)
print("Logits shape:", outputs.logits.shape)


## 20. Optional: Manual training loop structure

This cell shows the structure of a manual PyTorch fine-tuning loop.

It is commented out because it can take time. For most students, the `Trainer` API is the recommended first approach.


In [ ]:
# from torch.optim import AdamW
# from transformers import get_scheduler
# from tqdm.auto import tqdm

# manual_model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# optimizer = AdamW(manual_model.parameters(), lr=5e-5)

# num_epochs = 1
# num_training_steps = num_epochs * len(train_dataloader)

# lr_scheduler = get_scheduler(
#     "linear",
#     optimizer=optimizer,
#     num_warmup_steps=0,
#     num_training_steps=num_training_steps,
# )

# device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# manual_model.to(device)

# progress_bar = tqdm(range(num_training_steps))

# manual_model.train()
# for epoch in range(num_epochs):
#     for batch in train_dataloader:
#         batch = {k: v.to(device) for k, v in batch.items()}
#         outputs = manual_model(**batch)
#         loss = outputs.loss
#         loss.backward()

#         optimizer.step()
#         lr_scheduler.step()
#         optimizer.zero_grad()
#         progress_bar.update(1)


## 21. Summary

The fine-tuning workflow is:

```text
Load dataset
    ↓
Inspect labels and columns
    ↓
Load tokenizer
    ↓
Tokenize sentence pairs
    ↓
Use dynamic padding
    ↓
Load pretrained model with classification head
    ↓
Fine-tune with Trainer
    ↓
Evaluate with accuracy and F1
    ↓
Save model and tokenizer
    ↓
Use pipeline for inference
```

Key concepts:

| Concept | Meaning |
|---|---|
| GLUE | Benchmark collection for language understanding tasks |
| MRPC | Sentence-pair paraphrase classification dataset |
| Tokenizer | Converts text into model inputs |
| `token_type_ids` | Tells BERT which tokens belong to sentence 1 or sentence 2 |
| Dynamic padding | Pads only within each batch |
| `Trainer` | High-level Hugging Face training API |
| Fine-tuning | Continue training a pretrained model on a task-specific dataset |
| Logits | Raw model scores before softmax |
| Accuracy | Percentage of correct predictions |
| F1 | Balance between precision and recall |
